In [0]:
%sql
DROP TABLE IF EXISTS la_lakehouse.silver.la_building_permits_issued

In [0]:
%pip install shapely
%restart_python

In [0]:
from pyspark.sql.functions import col, trim 
from pyspark.sql import functions as F
from pyspark.sql.types import BinaryType
from shapely.wkt import loads
from itertools import chain

In [0]:
df = spark.table("la_lakehouse.bronze.la_building_permits_issued")

In [0]:
df.display()

#Converting The Data Types

In [0]:
@F.udf(returnType=BinaryType())
def wkt_to_wkb(wkt_str):
   if wkt_str is None:
      return None
   try:
      return loads(wkt_str).wkb
   except Exception as e:
      print(f"Error converting WKT to WKB: {e}")


timestamp_columns = ["submitted_date","issue_date","cofo_date","status_date","refresh_time"]
for c in timestamp_columns:
    df = df.withColumn(c,F.to_timestamp(col(c), "yyyy-MM-dd'T'HH:mm:ss.SSS"))
df = df.withColumn("_updated_at",F.to_timestamp(col("_updated_at"), "yyyy-MM-dd'T'HH:mm:ss.SSS'Z'"))


df = df.withColumn("geolocation",wkt_to_wkb(col("geolocation")))

df = df.withColumn("square_footage_raw", col("square_footage").cast("string"))
df = df.withColumn('square_footage_matches', F.regexp_extract_all(col("square_footage_raw"), F.lit(r"(-?\d+\.?\d*|-?\.\d+)")))
df = df.withColumn('square_footage', F.col("square_footage_matches")[0].cast("float"))
df = df.withColumn("square_footage_multi", F.when(F.size(col("square_footage_matches")) > 1, True).otherwise(False))


In [0]:
df.filter(col("square_footage_raw") == "-828").select("square_footage_raw", "square_footage_matches", "square_footage", "square_footage_multi").show(truncate=False)

In [0]:
for c in ["cd", "zip_code", "use_code", "du_changed", "valuation", "ct", "height", "lat", "lon"]:
    bad_count = df.filter(col(c).rlike("[^0-9.\\-]")).count()
    print(f"{c}: {bad_count} rows with non-numeric characters")

In [0]:
df.filter(col("square_footage_multi") == True).select("square_footage_raw", "square_footage", "square_footage_multi").show(n=df.count())

In [0]:
df.select("square_footage_raw").distinct().show(n=df.count())

In [0]:
df.printSchema()

# Fixing the multi-ct in ct

In [0]:
df = df.withColumn("ct_raw", col("ct").cast("string"))
df = df.withColumn('ct_split', F.split(col("ct_raw"), "[^0-9.]+"))
df = df.withColumn('ct', F.regexp_extract(col("ct_raw"), r"[0-9]+\.?[0-9]*", 0).cast("float"))
df = df.withColumn("ct_multi", F.when(F.size(col("ct_split")) > 1, True).otherwise(False))

In [0]:
df.filter(col("ct_multi") == True).select("ct_raw", "ct", "ct_multi").show(5, truncate=False)

In [0]:
df.select("ct").distinct().show(n=df.count())

In [0]:
columns_to_cast = {
   "cd":"int",
   "zip_code":"int",
   "use_code":"int",
   "du_changed":"int",
   "valuation":"int",
   "height":"float",
   "lat":"float",
   "lon":"float",
}


df = df.select([
   F.regexp_replace(col(c), ",", "").cast("float").cast("int").alias(c) if c in ["cd", "zip_code","use_code","du_changed","valuation",] else
   F.regexp_replace(col(c), ",", "").cast(columns_to_cast[c]).alias(c) if c in columns_to_cast else col(c)
   for c in df.columns 
])


In [0]:
df.printSchema()

In [0]:
df.display()

#Zone_Base And Zone_Is_Multi Extraction

In [0]:
df = df.withColumn("zone_raw",col("zone"))
df = df.withColumn("zone_base",F.regexp_replace(col("zone_raw"),r"^\(.*?\)|\(.*?\)|\[.*?\]",""))

In [0]:
df = df.withColumn("zone_split", F.split(col("zone_base"),","))
df = df.withColumn("zone", F.col("zone_split")[0])
df = df.withColumn("zone_is_multi", F.when(F.size(col("zone_split")) > 1, True).otherwise(False))

In [0]:
pattern = r"^([A-Za-z]+)-?\d"

df = df.withColumn(
    "zone", 
    F.when(
        F.col("zone_base").rlike(pattern),
         F.regexp_extract(col("zone_base"),pattern, 1)
    ).otherwise(F.lit("Specific Plan / Overlay"))
)

In [0]:
df.show(10)

#Cleaning CPA 

In [0]:
df = df.withColumn("cpa", F.regexp_replace(col("cpa"), "-", " - "))
df = df.withColumn("cpa", F.initcap(col("cpa")))
df = df.withColumn("cpa", F.regexp_replace(col("cpa"), "\\s+", " "))
df = df.withColumn("cpa", F.trim(col("cpa")))

In [0]:
df = df.withColumn("cpa", F.split(col("cpa"), ",")[0])
df = df.withColumn("cpa", F.trim(col("cpa")))

In [0]:
cpa_mapping = {
    "Canoga Park - West Hills - Winnetka - Woodland Hill": "Canoga Park - West Hills - Winnetka - Woodland Hills",
    "Mission Hills - North Hills - Panorama City": "Mission Hills - Panorama City - North Hills",
    "North Hollywood - Vally Village": "North Hollywood - Valley Village",
    "North East Los Angeles": "Northeast Los Angeles",
    "Palms - Del Rey - Mar Vista": "Palms - Mar Vista - Del Rey",
    "Palms Mar Vista Del Rey": "Palms - Mar Vista - Del Rey",
    "Silverlake - Echo Park": "Silver Lake - Echo Park",
    "Southeast La": "Southeast Los Angeles",
    "Sunland - La Tuna Cyn - Lakevw Ter - Shadow Hls - Tuj": "Sunland - E Latuna Cyn - Lakevw Ter - Shadow Hls - Tujunga",
    "Sunland - E Latuna Cyn - Lakevw Ter - Shadow Hls - Tu": "Sunland - E Latuna Cyn - Lakevw Ter - Shadow Hls - Tujunga",
    "Sunland - E Latuna Cyn - Lakevw Ter - Shadow Hls": "Sunland - E Latuna Cyn - Lakevw Ter - Shadow Hls - Tujunga",
    "Sherman Oaks - Studio City - Toluca Lake - Cahuenga Pass": "Sherman Oaks - Studio City - Toluca Lake - Cahuenga",
    "West Adams - Baldwin Park - Leimert Park": "West Adams - Baldwin Hills - Leimert",
}

In [0]:
mapping_expr = F.create_map([F.lit(x) for x in chain(*cpa_mapping.items())])

df = df.withColumn(
    "cpa",
    F.coalesce(mapping_expr[col("cpa")], col("cpa"))
)

In [0]:
df.select("cpa").distinct().orderBy("cpa").show(100, truncate=False)

#Normalizing Cnc

In [0]:
unique_df = df.select("cnc").distinct()
unique_df.show(n=unique_df.count(), truncate=False)

In [0]:
df = df.withColumn("cnc_raw",col("cnc"))
df = df.withColumn("cnc_split",F.split(col("cnc_raw"),","))
df = df.withColumn("cnc_split",F.transform(col("cnc_split"), lambda x: F.trim(x)))
df = df.withColumn("cnc_split",F.transform(col("cnc_split"), lambda x :F.trim(F.regexp_replace(x, r"(?i)\s+\b(NC|NDC|CC)\b\s*$",""))))
df = df.withColumn("cnc_split",F.array_distinct(col("cnc_split")))

In [0]:
df = df.withColumn("cnc", F.col("cnc_split")[0])
df = df.withColumn("cnc_is_multi", F.when(F.size(col("cnc_split")) > 1, True).otherwise(False))
df = df.withColumn("cnc",F.upper(col("cnc")))

In [0]:
df = df.withColumn("cnc", F.when(col("cnc") == 'LIMCOLN HEIGHTS',"LINCOLN HEIGHTS").otherwise(col("cnc")))
df = df.withColumn("cnc", F.when(col("cnc") == 'MID-TOWN NORTH HOLLYWOD',"MID-TOWN NORTH HOLLYWOOD").otherwise(col("cnc")))
df = df.withColumn("cnc", F.when(F.trim(F.upper(col("cnc"))) == "NULL", None).otherwise(col("cnc")))


In [0]:
df = df.withColumn("cnc", F.when(col("cnc").isin(
    ["WESTCHESTER/PLAYA", "WESTCHESTER/PLAYA DEL REY", "WESTCHESTER - PLAYA DEL REY", "WESTCHESTER-PLAYA DEL REY"]
), "WESTCHESTER/PLAYA DEL REY").otherwise(col("cnc")))

df = df.withColumn("cnc", F.when(col("cnc").isin(
    ["BEL AIR-BEVERLY CREST", "BEL AIR - BEVERLY CREST"]
), "BEL AIR-BEVERLY CREST").otherwise(col("cnc")))

df = df.withColumn("cnc", F.when(col("cnc").isin(
    ["WILSHIRE CENTER - KOREATOWN", "WILSHIRE CENTER-KOREATOWN", "WILSHIRE-CENTER-KOREATOWN"]
), "WILSHIRE-CENTER-KOREATOWN").otherwise(col("cnc")))

df = df.withColumn("cnc", F.when(col("cnc").isin(
    ["SUNLAND-TUJUNGA", "SUNLAND - TUJUNGA"]
), "SUNLAND-TUJUNGA").otherwise(col("cnc")))

df = df.withColumn("cnc", F.when(col("cnc").isin(
    ["WOODLAND HILLS-WARNER CENTER", "WOODLAND HILLS - WARNER CENTER"]
), "WOODLAND HILLS-WARNER CENTER").otherwise(col("cnc")))

df = df.withColumn("cnc", F.when(col("cnc").isin(
    ["ZAPATA-KING", "ZAPATA KING"]
), "ZAPATA-KING").otherwise(col("cnc")))

df = df.withColumn("cnc", F.when(col("cnc").isin(
    ["LA-32", "LA32"]
), "LA-32").otherwise(col("cnc")))

df = df.withColumn("cnc", F.when(col("cnc").isin(
    ["EMPOWERMENT CONGRESS NORTH", "EMPOWERMENT CONGRESS NORTH AREA"]
), "EMPOWERMENT CONGRESS NORTH AREA").otherwise(col("cnc")))

df = df.withColumn("cnc", F.when(col("cnc").isin(
    ["EMPOWERMENT CONGRESS WEST", "EMPOWERMENT CONGRESS WEST AREA"]
), "EMPOWERMENT CONGRESS WEST AREA").otherwise(col("cnc")))

df = df.withColumn("cnc", F.when(col("cnc").isin(
    ["EMPOWERMENT CONGRESS CENTRAL", "EMPOWERMENT CONGRESS CENTRAL AREA"]
), "EMPOWERMENT CONGRESS CENTRAL AREA").otherwise(col("cnc")))

df = df.withColumn("cnc", F.when(col("cnc").isin(
    ["EMPOWERMENT CONGRESS SOUTHEAST", "EMPOWERMENT CONGRESS SOUTHEAST AREA"]
), "EMPOWERMENT CONGRESS SOUTHEAST AREA").otherwise(col("cnc")))

df = df.withColumn("cnc", F.when(col("cnc").isin(
    ["EMPOWERMENT CONGRESS SOUTHWEST", "EMPOWERMENT CONGRESS SOUTHWEST AREA"]
), "EMPOWERMENT CONGRESS SOUTHWEST AREA").otherwise(col("cnc")))

df = df.withColumn("cnc", F.when(col("cnc").isin(
    ["UNTD NGHBRHDS OF HSTRC ARLINGTON HGHTS/W ADAM"]
), "UNTD NGHBRHDS OF HSTRC ARLINGTON HGHTS/W ADAMS/JEF").otherwise(col("cnc")))

df = df.withColumn("cnc", F.when(col("cnc").isin(
    ["COMMUNITY AND NEIGHBORS FOR NINTH DISTRICT UN",
     "COMMUNITY AND NEIGHBORS FOR NINTH DISTRICT UNITY (CANNDU)",
     "COMMUNITY AND NEIGHBORS FOR NINTH DISTRICT UNITY ("]
), "COMMUNITY AND NEIGHBORS FOR NINTH DISTRICT UNITY").otherwise(col("cnc")))

df = df.withColumn("cnc", F.when(col("cnc").isin(
    ["VOICES", "VOICES OF 90037"]
), "VOICES").otherwise(col("cnc")))

df = df.withColumn("cnc", F.when(col("cnc").isin(
    ["NORTH HOLLYWOOD NORTH EAST", "NORTH HOLLYWOOD NORTHEAST"]
), "NORTH HOLLYWOOD NORTHEAST").otherwise(col("cnc")))

In [0]:
df.filter(col("cnc_raw") == "Echo Park NC, Historic Cultural North").select("cnc_raw", "cnc_split", "cnc", "cnc_is_multi").show(truncate=False)

In [0]:
df.select("cnc").distinct().orderBy("cnc").show(400, truncate=False)

In [0]:
unique_df = df.select("cnc").distinct()
unique_df.show(n=unique_df.count(), truncate=False)

#Adu_Changed Outlier Cutoff

In [0]:
df = df.withColumn("adu_changed_raw", col("adu_changed"))
df = df.withColumn(
    "adu_changed",
    F.when(F.col("adu_changed_raw").between(-5, 20), F.col("adu_changed_raw"))
     .otherwise(F.lit(None))
)

In [0]:
df.display()

#Getting Rid Of Whitespace

In [0]:
df = df.withColumn("pin_nbr",F.regexp_replace(col("pin_nbr"), r"\s+", ""))

In [0]:
df.show(10)

#Normalizing columns

In [0]:
# EV and Solar N -> No and Y -> Yes 

df = (
    df
    .withColumn(
        "ev",
        F.when(F.upper(F.col("ev")) == "N", "No")
        .when(F.upper(F.col("ev")) == "Y", "Yes")
    )
    .withColumn(
        "solar",
        F.when(F.upper(F.col("solar")) == "N", "No")
        .when(F.upper(F.col("solar")) == "Y", "Yes")
    )
    
)

In [0]:
df.display()

# Dropping Unwanted Columns

In [0]:
df = df.drop("zone_split","ct_split")

In [0]:
df.printSchema()

#TESTING 

In [0]:
df.filter(~col("adu_changed_raw").between(-5, 20)).select("adu_changed_raw", "adu_changed").show(10)

In [0]:
df.filter(col("zone_raw") == "[T][Q]C2-1VL").select("zone_raw", "zone_base").show(truncate=False)

In [0]:
df.filter(col("zone_raw").rlike(",")).select("zone_raw", "zone_base", "zone", "zone_is_multi").show(10, truncate=False)

In [0]:
df.filter(col("zone_base") == "LASED").select("zone_raw", "zone_base", "zone").show(truncate=False)

In [0]:
%sql
SELECT COUNT(*)
FROM la_lakehouse.bronze.la_building_permits_issued;

In [0]:
df.count()

In [0]:
df.printSchema()

In [0]:
df.show(20)

In [0]:
df.filter(col("cnc_raw") == 'Arroyo Seco, Glassell Park').select("cnc_raw", "cnc", "cnc_is_multi").show(5, truncate=False)

#Write To Silver Table

In [0]:
(
    df.write
    .mode("overwrite")
    .format("delta")
    .saveAsTable("la_lakehouse.silver.la_building_permits_issued")
)

#Quick-Look 

In [0]:
%sql
SELECT * 
FROM la_lakehouse.silver.la_building_permits_issued
LIMIT 30;

In [0]:
%sql
SELECT DISTINCT cpa FROM la_lakehouse.silver.silver_permits_issued_enriched ORDER BY cpa